# ERIS drift detector — Colab demo

This notebook installs the repo, exercises the new drift-detector features on synthetic SAE outputs, and saves JSON reports you can inspect in Colab.

Features covered:
- `layer_weights` for weighted aggregation
- `comparison_mode="previous"` for step-to-step drift
- `severity`, `layers_ranked`, and `to_dict()` for reporting
- labeled feature refs in summaries via `active_feature_labels`


In [ ]:
import os
from pathlib import Path

REPO_URL = os.environ.get('LATENT_RELAY_REPO_URL', 'https://github.com/ArthurVigier/latent-relay.git')
REPO_DIR = Path('/content/latent-relay')

if not REPO_DIR.exists():
    !git clone $REPO_URL /content/latent-relay

%cd /content/latent-relay
!python -m pip install -q --upgrade pip
!python -m pip install -q numpy matplotlib pytest torch


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from eris.drift_detector import DriftDetector
from eris.sae_probe import ProbeOutput


In [ ]:
def make_probe_output(layer, active, raw, labels=None):
    labels = list(labels) if labels is not None else [None] * len(active)
    return ProbeOutput(
        layer=layer,
        active_feature_indices=list(active)[:5],
        active_feature_values=[1.0] * min(len(active), 5),
        all_active_indices=list(active),
        n_active=len(active),
        raw_activations=np.asarray(raw, dtype=np.float32),
        elapsed_s=0.0,
        active_feature_labels=labels[:5],
    )

def make_state(spec):
    return {layer: make_probe_output(layer, active, raw, labels) for layer, (active, raw, labels) in spec.items()}


In [ ]:
detector = DriftDetector(
    threshold=0.30,
    window=2,
    jaccard_weight=0.7,
    cosine_weight=0.3,
    layer_weights={10: 1.0, 20: 2.5, 30: 1.5},
    comparison_mode='previous',
)

reference = make_state({
    10: ([1, 2, 3], [1.0, 0.0, 0.0, 0.0], ['setup', 'algebra', 'baseline']),
    20: ([10, 11, 12], [0.0, 1.0, 0.0, 0.0], ['proof_state', 'factoring', 'symmetry']),
    30: ([20, 21, 22], [0.0, 0.0, 1.0, 0.0], ['verification', 'conclusion', 'consistency']),
})
detector.register_reference(reference)

states = [
    make_state({
        10: ([1, 2, 4], [0.9, 0.1, 0.0, 0.0], ['setup', 'algebra', 'branching']),
        20: ([10, 90, 91], [0.0, 0.4, 0.6, 0.0], ['proof_state', 'modular_reasoning', 'case_split']),
        30: ([20, 21, 22], [0.0, 0.0, 1.0, 0.0], ['verification', 'conclusion', 'consistency']),
    }),
    make_state({
        10: ([1, 2, 4], [0.9, 0.1, 0.0, 0.0], ['setup', 'algebra', 'branching']),
        20: ([90, 91, 92], [0.0, 0.1, 0.9, 0.0], ['modular_reasoning', 'case_split', 'residue_tracking']),
        30: ([20, 25, 26], [0.0, 0.0, 0.5, 0.5], ['verification', 'contradiction', 'summary']),
    }),
    make_state({
        10: ([1, 2, 4], [0.9, 0.1, 0.0, 0.0], ['setup', 'algebra', 'branching']),
        20: ([90, 91, 92], [0.0, 0.1, 0.9, 0.0], ['modular_reasoning', 'case_split', 'residue_tracking']),
        30: ([20, 25, 26], [0.0, 0.0, 0.5, 0.5], ['verification', 'contradiction', 'summary']),
    }),
]

reports = [detector.compute_drift(state, step=i + 1) for i, state in enumerate(states)]
payload = [report.to_dict() for report in reports]
print(reports[-1].summary)
payload


In [ ]:
steps = [item['step'] for item in payload]
raw_scores = [item['raw_drift_score'] for item in payload]
smooth_scores = [item['drift_score'] for item in payload]

plt.figure(figsize=(8, 4))
plt.plot(steps, raw_scores, marker='o', label='raw drift')
plt.plot(steps, smooth_scores, marker='s', label='smoothed drift')
plt.axhline(0.30, color='red', linestyle='--', label='threshold')
plt.title('Synthetic ERIS drift trajectory')
plt.xlabel('step')
plt.ylabel('score')
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.legend()
plt.show()


In [ ]:
out_path = Path('/content/drift_demo_reports.json')
out_path.write_text(json.dumps(payload, indent=2), encoding='utf-8')
print(f'Saved: {out_path}')
print('Last summary with labels:')
print(reports[-1].summary)
print('Serialized labels:', payload[-1].get('feature_labels'))


In [ ]:
!pytest -q test_drift_detector.py
